# Colab H200 开源声音模仿与视频配音工作台（`colab-voice-dubbing`）

支持 3 个顶级开源 TTS（各自独立 `uv` 虚拟环境，互不冲突；大权重与缓存持久化在 Google Drive）：
- **IndexTTS-2.5**：剧情配音主力（音色与情绪解耦、支持情绪迁移、原生精准卡时长）
- **Fish Audio S2 Pro**：拟人度与自然长旁白天花板（需先在 Hugging Face 同意 `fishaudio/s2-pro` 协议并配置 Secrets `HF_TOKEN`）
- **VoxCPM 2**：无参考音频时先用中文“捏声线”或出 48kHz 方言母本

> **省算力提示**：首次下载权重建议先用**免费 CPU 运行时**跑完权重下载，再切换 **H200** 运行对应模型 Cell；配音结束并存盘后请立即「断开连接并删除运行时」。

## Cell 1：挂载 Google Drive & 基础环境（每次重开 Colab 必跑）

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")

# 持久化目录（存在你的 Google Drive 里）
DRIVE_ROOT = "/content/drive/MyDrive/tts_studio"
MODELS_DIR = f"{DRIVE_ROOT}/models"
HF_CACHE   = f"{DRIVE_ROOT}/hf_cache"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs"

for d in [MODELS_DIR, HF_CACHE, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE

# 如需下载 Fish Audio S2 Pro（受限仓库），先在左侧 🔑 Secrets 添加 HF_TOKEN
try:
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("✅ 已加载 HF_TOKEN")
except Exception:
    print("ℹ️ 未设置 HF_TOKEN（跑 IndexTTS-2.5 / VoxCPM 2 不需要；跑 Fish S2 Pro 时再加即可）")

!pip install -q -U uv huggingface_hub
print("✅ 初始化完成，目录：", DRIVE_ROOT)


## Cell 2：主力 1 — `IndexTTS-2.5`（控情绪 + 卡视频时长首选）

In [ ]:
import os
from huggingface_hub import snapshot_download
from google.colab import output

INDEX_REPO = "/content/index-tts"
INDEX_CKPT = f"{MODELS_DIR}/IndexTTS-2.5"

# 1. 克隆并安装到独立 uv 环境（约 1 分钟）
if not os.path.exists(INDEX_REPO):
    !git lfs install
    !git clone https://github.com/index-tts/index-tts.git {INDEX_REPO}
    !cd {INDEX_REPO} && git lfs pull && uv sync --all-extras

# 2. 下载权重到 Google Drive（已有则自动跳过）
if not os.path.exists(f"{INDEX_CKPT}/config.yaml"):
    snapshot_download("IndexTeam/IndexTTS-2.5", local_dir=INDEX_CKPT)

# 3. 弹出 Colab 专属网页入口并启动 WebUI
print("👇 等下方出现 Running on http://0.0.0.0:7860 后，点击此链接打开界面：")
output.serve_kernel_port_as_window(7860)

!cd {INDEX_REPO} && uv run webui.py --model_dir "{INDEX_CKPT}" --host 0.0.0.0 --port 7860


## Cell 3：主力 2 — `Fish Audio S2 Pro`（拟人度与自然旁白天花板）

> 前置一步：先在 [huggingface.co/fishaudio/s2-pro](https://huggingface.co/fishaudio/s2-pro) 同意协议，并将 Read Token 填入 Colab 左侧 **Secrets (`HF_TOKEN`)** 后重跑 Cell 1。

In [ ]:
import os
from huggingface_hub import snapshot_download
from google.colab import output

FISH_REPO = "/content/fish-speech"
FISH_VENV = "/content/envs/fish"
FISH_CKPT = f"{MODELS_DIR}/fish-s2-pro"

# 1. 克隆并安装到独立 uv 虚拟环境
if not os.path.exists(FISH_REPO):
    !git clone https://github.com/fishaudio/fish-speech.git {FISH_REPO}
    !uv venv {FISH_VENV} --python 3.11
    !uv pip install --python {FISH_VENV}/bin/python -e {FISH_REPO}

# 2. 下载 S2 Pro 权重到 Google Drive（需已配置 HF_TOKEN）
if not os.path.exists(f"{FISH_CKPT}/config.json"):
    assert os.environ.get("HF_TOKEN"), "请先在 Colab 左侧 Secrets 配置 HF_TOKEN 并同意 fishaudio/s2-pro 协议"
    snapshot_download(
        "fishaudio/s2-pro",
        local_dir=FISH_CKPT,
        token=os.environ.get("HF_TOKEN"),
    )

# 3. 打开端口并启动 WebUI（--compile 首次编译约 1 分钟，之后在 H200 上秒出）
print("👇 等待启动完成后，点击此链接打开 Fish Audio WebUI：")
output.serve_kernel_port_as_window(7860)

!cd {FISH_REPO} && {FISH_VENV}/bin/python -m tools.run_webui \
    --llama-checkpoint-path "{FISH_CKPT}" \
    --decoder-checkpoint-path "{FISH_CKPT}/codec.pth" \
    --compile


## Cell 4：可选起手工具 — `VoxCPM 2`（文字捏原创声线 / 48kHz 方言母本）

In [ ]:
import os
from google.colab import output

VOX_REPO = "/content/VoxCPM"
VOX_VENV = "/content/envs/voxcpm"

if not os.path.exists(VOX_REPO):
    !git clone https://github.com/OpenBMB/VoxCPM.git {VOX_REPO}
    !uv venv {VOX_VENV} --python 3.11
    !uv pip install --python {VOX_VENV}/bin/python -e {VOX_REPO}

print("👇 首次启动会自动将 openbmb/VoxCPM2 缓存到 Google Drive，完成后点此打开：")
output.serve_kernel_port_as_window(7860)

!cd {VOX_REPO} && {VOX_VENV}/bin/python app.py


## 备用 Cell：`cloudflared` 公网临时隧道（端口白屏时使用）

在启动任意 WebUI Cell **之前**运行此格获取 `*.trycloudflare.com` 公网地址，等 WebUI 就绪后刷新即可。

In [ ]:
!test -x /usr/local/bin/cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)
!pkill -f "cloudflared tunnel" || true
!nohup cloudflared tunnel --url http://127.0.0.1:7860 > /tmp/cf.log 2>&1 &
!sleep 4 && grep -o 'https://[^ ]*trycloudflare.com' /tmp/cf.log | head -n 1
